In [1]:
"""
EVALUATE SAVED MODELS - COMPREHENSIVE METRICS
==============================================
This script loads pre-trained models (.h5 files) and evaluates them on test data
to compute per-class and overall precision, recall, F1-score, Dice, and IoU.

Uses the same data split (seed=42) as the training script to reproduce exact test data.
"""

import os
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import models, backend as K
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

print(f"\n{'='*100}")
print(f"{'EVALUATING SAVED MODELS - COMPREHENSIVE METRICS':^100s}")
print(f"{'='*100}\n")

# ============================================================================
# CONFIGURATION
# ============================================================================

# Paths
ROOT_DIR = "/kaggle/input/refuge/REFUGE/"
MODELS_DIR = "/kaggle/input/mobilenet-models/tensorflow2/default/1/"
OUTPUT_DIR = "evaluation_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Model files to evaluate
MODEL_FILES = [
    "mobilenet_unet_cbam.h5",
    "mobilenet_unet_transformer.h5",
    "mobilenet_unet_convex_prior.h5",
    "mobilenet_unet_focal_eiou.h5",
    "mobilenet_unet_fpn.h5",
    "mobilenet_unet_mc_dropout.h5",
    "mobilenet_unet_deeper.h5",
    "2_Deeper_MC_Dropout.h5",
    "3_Focal_EIoU_Deeper.h5",
    "4_FULL_Combination.h5"
]

MODEL_NAMES = [
    "MobileNet-UNet + CBAM",
    "MobileNet-UNet + Transformer",
    "MobileNet-UNet + Convex Prior",
    "MobileNet-UNet + Focal-EIoU",
    "MobileNet-UNet + FPN",
    "MobileNet-UNet + MC Dropout",
    "MobileNet-UNet + Deeper",
    "Deeper + MC Dropout",
    "Focal-EIoU + Deeper",
    "FULL Combination (MC+Focal+Deeper)"
]

# ============================================================================
# GPU CONFIGURATION
# ============================================================================

def configure_gpu():
    """Configure GPU settings"""
    print("Configuring GPU...")
    physical_devices = tf.config.list_physical_devices('GPU')
    if physical_devices:
        try:
            for gpu in physical_devices:
                tf.config.experimental.set_memory_growth(gpu, True)
            tf.config.set_visible_devices(physical_devices[0], 'GPU')
            print(f"✓ GPU found: {physical_devices[0].name}")
            return True
        except RuntimeError as e:
            print(f"✗ GPU configuration error: {e}")
            return False
    else:
        print("✗ No GPU found, using CPU")
        return False

# ============================================================================
# DATA LOADING (SAME AS TRAINING SCRIPT)
# ============================================================================

def load_images_masks(images_folder, masks_folder, img_size=(256, 256)):
    """Load images and masks from folders"""
    images, masks = [], []
    image_files = sorted(os.listdir(images_folder))
    mask_files = sorted(os.listdir(masks_folder))
    mask_dict = {os.path.splitext(f)[0]: f for f in mask_files}
    
    for img_file in image_files:
        base = os.path.splitext(img_file)[0]
        if base in mask_dict:
            img = cv2.imread(os.path.join(images_folder, img_file))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, img_size) / 255.0
            
            mask = cv2.imread(os.path.join(masks_folder, mask_dict[base]), cv2.IMREAD_GRAYSCALE)
            mask = cv2.resize(mask, img_size)
            unique_vals = np.unique(mask)
            if len(unique_vals) > 3 or np.max(unique_vals) > 2:
                mask = np.clip(mask // 85, 0, 2).astype(np.uint8)
            
            images.append(img)
            masks.append(mask)
    
    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.uint8)

def load_all_data(root_dir):
    """Load all data from existing splits and merge"""
    all_images, all_masks = [], []
    for split in ["train", "val", "test"]:
        print(f"  Loading {split} data...")
        imgs, msks = load_images_masks(
            os.path.join(root_dir, split, "Images"),
            os.path.join(root_dir, split, "Masks")
        )
        all_images.append(imgs)
        all_masks.append(msks)
        print(f"    {split}: {imgs.shape[0]} samples")
    
    all_images = np.concatenate(all_images, axis=0)
    all_masks = np.concatenate(all_masks, axis=0)
    print(f"  Total: {all_images.shape[0]} samples\n")
    return all_images, all_masks

def create_test_split(all_images, all_masks):
    """
    Create the SAME test split as training script using seed=42
    Split: 600 train, 200 val, 400 test
    """
    # First split: separate 400 test samples
    _, X_test, _, y_test = train_test_split(
        all_images, all_masks, test_size=400, random_state=42
    )
    
    print(f"Test split created: {X_test.shape[0]} samples")
    print(f"  Shape: {X_test.shape}")
    print(f"  Masks shape: {y_test.shape}")
    print(f"  Mask classes: {np.unique(y_test)}\n")
    
    return X_test, y_test

# ============================================================================
# CUSTOM LAYERS (FOR MODEL LOADING)
# ============================================================================

from tensorflow.keras import layers

class SimpleTransformerBlock(layers.Layer):
    """Simple Vision Transformer Block for UNet enhancement"""
    
    def __init__(self, num_heads=4, ff_dim=256, dropout_rate=0.1, name="simple_transformer", **kwargs):
        super(SimpleTransformerBlock, self).__init__(name=name, **kwargs)
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        self.dropout_rate = dropout_rate
        
    def build(self, input_shape):
        self.channels = input_shape[-1]
        self.height = input_shape[1]
        self.width = input_shape[2]
        self.seq_len = self.height * self.width
        
        # Build transformer layers
        self.reshape_to_seq = layers.Reshape((-1, self.channels))
        
        # Positional encoding (learnable)
        self.pos_embedding = layers.Embedding(
            input_dim=self.seq_len, 
            output_dim=self.channels,
            embeddings_initializer='uniform'
        )
        
        # Multi-head attention
        self.mha = layers.MultiHeadAttention(
            num_heads=self.num_heads,
            key_dim=self.channels // self.num_heads,
            dropout=self.dropout_rate
        )
        
        # Layer normalization and dropout
        self.ln1 = layers.LayerNormalization(epsilon=1e-6)
        self.ln2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(self.dropout_rate)
        self.dropout2 = layers.Dropout(self.dropout_rate)
        
        # Feed forward network
        self.ffn = tf.keras.Sequential([
            layers.Dense(self.ff_dim, activation='relu'),
            layers.Dropout(self.dropout_rate),
            layers.Dense(self.channels)
        ])
        
        # Reshape back to spatial
        self.reshape_to_spatial = layers.Reshape((self.height, self.width, self.channels))
        
        super(SimpleTransformerBlock, self).build(input_shape)
    
    def call(self, x, training=None):
        batch_size = tf.shape(x)[0]
        
        # Flatten spatial dimensions to sequence
        x_seq = self.reshape_to_seq(x)
        
        # Add positional encoding
        positions = tf.range(start=0, limit=self.seq_len, delta=1)
        positions = tf.expand_dims(positions, 0)
        positions = tf.tile(positions, [batch_size, 1])
        pos_encodings = self.pos_embedding(positions)
        
        x_seq = x_seq + pos_encodings
        
        # Multi-head self-attention with residual connection
        attn_input = self.ln1(x_seq)
        attn_output = self.mha(attn_input, attn_input, training=training)
        attn_output = self.dropout1(attn_output, training=training)
        x_seq = x_seq + attn_output  # Residual connection
        
        # Feed forward network with residual connection
        ffn_input = self.ln2(x_seq)
        ffn_output = self.ffn(ffn_input, training=training)
        ffn_output = self.dropout2(ffn_output, training=training)
        x_seq = x_seq + ffn_output  # Residual connection
        
        # Reshape back to spatial format
        output = self.reshape_to_spatial(x_seq)
        
        return output
    
    def get_config(self):
        config = super(SimpleTransformerBlock, self).get_config()
        config.update({
            'num_heads': self.num_heads,
            'ff_dim': self.ff_dim,
            'dropout_rate': self.dropout_rate
        })
        return config

class SpatialAttention(layers.Layer):
    """Lightweight spatial attention mechanism with channel and spatial attention"""
    
    def __init__(self, name="spatial_attention", **kwargs):
        super(SpatialAttention, self).__init__(name=name, **kwargs)
        
    def build(self, input_shape):
        self.channels = input_shape[-1]
        
        # Channel attention components
        self.global_avg_pool = layers.GlobalAveragePooling2D(keepdims=True)
        self.global_max_pool = layers.GlobalMaxPooling2D(keepdims=True)
        
        self.fc1 = layers.Dense(self.channels // 8, activation='relu')
        self.fc2 = layers.Dense(self.channels, activation='sigmoid')
        
        # Spatial attention
        self.conv_spatial = layers.Conv2D(1, (7, 7), padding='same', activation='sigmoid')
        
        super(SpatialAttention, self).build(input_shape)
    
    def call(self, x, training=None):
        # Channel attention
        avg_pool = self.global_avg_pool(x)
        max_pool = self.global_max_pool(x)
        
        avg_out = self.fc2(self.fc1(layers.Flatten()(avg_pool)))
        max_out = self.fc2(self.fc1(layers.Flatten()(max_pool)))
        
        channel_attention = tf.nn.sigmoid(avg_out + max_out)
        channel_attention = tf.expand_dims(tf.expand_dims(channel_attention, 1), 1)
        
        x = x * channel_attention
        
        # Spatial attention
        avg_out = tf.reduce_mean(x, axis=-1, keepdims=True)
        max_out = tf.reduce_max(x, axis=-1, keepdims=True)
        spatial_input = tf.concat([avg_out, max_out], axis=-1)
        spatial_attention = self.conv_spatial(spatial_input)
        
        return x * spatial_attention
    
    def get_config(self):
        config = super(SpatialAttention, self).get_config()
        return config

# ============================================================================
# CUSTOM METRICS (FOR MODEL LOADING)
# ============================================================================

def dice_coef_multiclass(y_true, y_pred, smooth=1e-7):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def dice_coef_class(y_true, y_pred, class_index, smooth=1e-7):
    y_true_class = K.cast(K.equal(K.argmax(y_true, axis=-1), class_index), 'float32')
    y_pred_class = K.cast(K.equal(K.argmax(y_pred, axis=-1), class_index), 'float32')
    intersection = K.sum(y_true_class * y_pred_class)
    return (2. * intersection + smooth) / (K.sum(y_true_class) + K.sum(y_pred_class) + smooth)

def dice_class_0(y_true, y_pred): return dice_coef_class(y_true, y_pred, 0)
def dice_class_1(y_true, y_pred): return dice_coef_class(y_true, y_pred, 1)
def dice_class_2(y_true, y_pred): return dice_coef_class(y_true, y_pred, 2)

def iou_coef_multiclass(y_true, y_pred, smooth=1e-7):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    union = K.sum(y_true_f) + K.sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)

def iou_coef_class(y_true, y_pred, class_index, smooth=1e-7):
    y_true_class = K.cast(K.equal(K.argmax(y_true, axis=-1), class_index), 'float32')
    y_pred_class = K.cast(K.equal(K.argmax(y_pred, axis=-1), class_index), 'float32')
    intersection = K.sum(y_true_class * y_pred_class)
    union = K.sum(y_true_class) + K.sum(y_pred_class) - intersection
    return (intersection + smooth) / (union + smooth)

def iou_class_0(y_true, y_pred): return iou_coef_class(y_true, y_pred, 0)
def iou_class_1(y_true, y_pred): return iou_coef_class(y_true, y_pred, 1)
def iou_class_2(y_true, y_pred): return iou_coef_class(y_true, y_pred, 2)

def focal_loss(y_true, y_pred, gamma=2.0, alpha=None, epsilon=1e-7):
    """Focal loss for loading models trained with it"""
    if alpha is None:
        alpha = [0.25, 1.0, 1.0]
    
    y_pred = K.clip(y_pred, epsilon, 1.0 - epsilon)
    focal_loss_value = 0.0
    
    for c in range(3):
        y_true_c = y_true[:, :, :, c]
        y_pred_c = y_pred[:, :, :, c]
        pt = y_pred_c
        focal_weight = K.pow(1.0 - pt, gamma)
        cross_entropy = -K.log(pt)
        focal_loss_c = alpha[c] * focal_weight * cross_entropy * y_true_c
        focal_loss_value += K.mean(focal_loss_c)
    
    return focal_loss_value

def enhanced_iou_loss(y_true, y_pred, smooth=1e-7):
    """Enhanced IoU loss for loading models trained with it"""
    eiou_loss_value = 0.0
    
    for c in range(3):
        y_true_c = y_true[:, :, :, c]
        y_pred_c = y_pred[:, :, :, c]
        
        intersection = K.sum(y_true_c * y_pred_c, axis=[1, 2])
        union = K.sum(y_true_c, axis=[1, 2]) + K.sum(y_pred_c, axis=[1, 2]) - intersection
        iou = (intersection + smooth) / (union + smooth)
        
        height = K.cast(K.shape(y_true_c)[1], 'float32')
        width = K.cast(K.shape(y_true_c)[2], 'float32')
        
        y_coords = K.arange(0, height)
        x_coords = K.arange(0, width)
        y_grid = K.tile(K.reshape(y_coords, (-1, 1)), (1, K.cast(width, 'int32')))
        x_grid = K.tile(K.reshape(x_coords, (1, -1)), (K.cast(height, 'int32'), 1))
        y_grid = K.cast(y_grid, 'float32')
        x_grid = K.cast(x_grid, 'float32')
        
        true_mass = K.sum(y_true_c, axis=[1, 2]) + smooth
        true_center_y = K.sum(y_true_c * y_grid, axis=[1, 2]) / true_mass
        true_center_x = K.sum(y_true_c * x_grid, axis=[1, 2]) / true_mass
        
        pred_mass = K.sum(y_pred_c, axis=[1, 2]) + smooth
        pred_center_y = K.sum(y_pred_c * y_grid, axis=[1, 2]) / pred_mass
        pred_center_x = K.sum(y_pred_c * x_grid, axis=[1, 2]) / pred_mass
        
        center_distance = K.square(true_center_y - pred_center_y) + K.square(true_center_x - pred_center_x)
        diagonal = K.square(height) + K.square(width)
        center_penalty = center_distance / (diagonal + smooth)
        
        true_width = K.sqrt(K.sum(K.sum(y_true_c, axis=1), axis=1) + smooth)
        true_height = K.sqrt(K.sum(K.sum(y_true_c, axis=2), axis=1) + smooth)
        pred_width = K.sqrt(K.sum(K.sum(y_pred_c, axis=1), axis=1) + smooth)
        pred_height = K.sqrt(K.sum(K.sum(y_pred_c, axis=2), axis=1) + smooth)
        
        width_diff = K.square(pred_width - true_width)
        height_diff = K.square(pred_height - true_height)
        aspect_penalty = width_diff / (K.square(width) + smooth) + height_diff / (K.square(height) + smooth)
        
        eiou_c = 1.0 - iou + center_penalty + aspect_penalty
        eiou_loss_value += K.mean(eiou_c)
    
    return eiou_loss_value / 3.0

def focal_eiou_combined_loss(y_true, y_pred, focal_weight=1.0, eiou_weight=1.0, gamma=2.0, alpha=None):
    """Combined focal + EIoU loss"""
    fl = focal_loss(y_true, y_pred, gamma=gamma, alpha=alpha)
    eiou = enhanced_iou_loss(y_true, y_pred)
    return focal_weight * fl + eiou_weight * eiou

# ============================================================================
# COMPREHENSIVE METRICS CALCULATION
# ============================================================================

def calculate_per_class_metrics(y_true, y_pred, class_idx, class_name, smooth=1e-7):
    """
    Calculate precision, recall, F1-score, Dice, and IoU for a single class
    
    Args:
        y_true: Ground truth masks (H, W) with class labels
        y_pred: Predicted masks (H, W) with class labels
        class_idx: Index of the class to evaluate
        class_name: Name of the class (for display)
        smooth: Smoothing factor for numerical stability
    
    Returns:
        Dictionary with all metrics
    """
    # Create binary masks for the specific class
    y_true_binary = (y_true == class_idx).astype(np.float32).flatten()
    y_pred_binary = (y_pred == class_idx).astype(np.float32).flatten()
    
    # Precision, Recall, F1-score (using sklearn)
    precision = precision_score(y_true_binary, y_pred_binary, zero_division=0)
    recall = recall_score(y_true_binary, y_pred_binary, zero_division=0)
    f1 = f1_score(y_true_binary, y_pred_binary, zero_division=0)
    
    # Dice coefficient
    intersection = np.sum(y_true_binary * y_pred_binary)
    dice = (2.0 * intersection + smooth) / (np.sum(y_true_binary) + np.sum(y_pred_binary) + smooth)
    
    # IoU (Jaccard index)
    union = np.sum(y_true_binary) + np.sum(y_pred_binary) - intersection
    iou = (intersection + smooth) / (union + smooth)
    
    return {
        'class': class_name,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'dice': dice,
        'iou': iou
    }

def evaluate_model_comprehensive(model, X_test, y_test, model_name, batch_size=8):
    """
    Comprehensive evaluation of a model on test data
    
    Returns per-class and overall metrics
    """
    print(f"\n{'='*100}")
    print(f"EVALUATING: {model_name}")
    print(f"{'='*100}")
    
    # Model parameters
    total_params = model.count_params()
    print(f"  Total Parameters: {total_params:,}")
    
    # Get predictions
    print("  Generating predictions...")
    y_pred_probs = model.predict(X_test, verbose=0, batch_size=batch_size)
    y_pred_labels = np.argmax(y_pred_probs, axis=-1)  # (N, H, W)
    
    # Class names
    class_names = ["Background", "Disc", "Cup"]
    
    # Calculate metrics for each class
    all_metrics = []
    
    print(f"\nPrecision, Recall, F1 Score per class:")
    
    for class_idx, class_name in enumerate(class_names):
        metrics = calculate_per_class_metrics(y_test, y_pred_labels, class_idx, class_name)
        all_metrics.append(metrics)
        
        print(f"  {class_name}: Precision={metrics['precision']:.4f}, Recall={metrics['recall']:.4f}, F1={metrics['f1_score']:.4f}")
    
    # Calculate overall metrics (mean across 3 classes)
    overall_precision = np.mean([m['precision'] for m in all_metrics])
    overall_recall = np.mean([m['recall'] for m in all_metrics])
    overall_f1 = np.mean([m['f1_score'] for m in all_metrics])
    overall_dice = np.mean([m['dice'] for m in all_metrics])
    overall_iou = np.mean([m['iou'] for m in all_metrics])
    
    print(f"Overall Precision: {overall_precision:.4f}")
    print(f"Overall Recall: {overall_recall:.4f}")
    print(f"Overall F1 Score: {overall_f1:.4f}")
    
    print(f"\nDice and IoU per class:")
    for class_idx, class_name in enumerate(class_names):
        metrics = all_metrics[class_idx]
        print(f"  {class_name}: Dice={metrics['dice']:.4f}, IoU={metrics['iou']:.4f}")
    
    print(f"Overall Dice: {overall_dice:.4f}")
    print(f"Overall IoU: {overall_iou:.4f}")
    
    # Store overall metrics
    overall_metrics = {
        'class': 'Overall (Mean)',
        'precision': overall_precision,
        'recall': overall_recall,
        'f1_score': overall_f1,
        'dice': overall_dice,
        'iou': overall_iou
    }
    
    all_metrics.append(overall_metrics)
    
    return all_metrics, model.count_params()

# ============================================================================
# MAIN EVALUATION PIPELINE
# ============================================================================

def main():
    # Configure GPU
    configure_gpu()
    
    # Load test data (using same seed=42)
    print(f"\n{'='*100}")
    print(f"{'LOADING TEST DATA':^100s}")
    print(f"{'='*100}\n")
    
    all_images, all_masks = load_all_data(ROOT_DIR)
    X_test, y_test = create_test_split(all_images, all_masks)
    
    # Custom Lambda functions for CBAM (with output_shape specified)
    from functools import wraps
    
    def create_lambda_with_output_shape(func, output_shape_func=None):
        """Helper to create Lambda layers with output_shape"""
        if output_shape_func is None:
            # For reduce operations that keep spatial dims but change channels
            output_shape_func = lambda input_shape: (*input_shape[:-1], 1)
        return layers.Lambda(func, output_shape=output_shape_func)
    
    # Custom objects for model loading
    custom_objects = {
        'dice_coef_multiclass': dice_coef_multiclass,
        'dice_class_0': dice_class_0,
        'dice_class_1': dice_class_1,
        'dice_class_2': dice_class_2,
        'iou_coef_multiclass': iou_coef_multiclass,
        'iou_class_0': iou_class_0,
        'iou_class_1': iou_class_1,
        'iou_class_2': iou_class_2,
        'focal_loss': focal_loss,
        'enhanced_iou_loss': enhanced_iou_loss,
        'focal_eiou_combined_loss': focal_eiou_combined_loss,
        'SimpleTransformerBlock': SimpleTransformerBlock,
        'SpatialAttention': SpatialAttention
    }
    
    # Evaluate all models
    all_results = {}
    all_params = {}
    
    for model_file, model_name in zip(MODEL_FILES, MODEL_NAMES):
        model_path = os.path.join(MODELS_DIR, model_file)
        
        if not os.path.exists(model_path):
            print(f"\n✗ Model not found: {model_path}")
            continue
        
        print(f"\n{'='*100}")
        print(f"LOADING MODEL: {model_file}")
        print(f"{'='*100}")
        
        # Clear session
        K.clear_session()
        
        # Load model
        try:
            # Attempt to load model with custom objects
            with tf.keras.utils.custom_object_scope(custom_objects):
                model = models.load_model(model_path, compile=False)
            print(f"✓ Model loaded successfully")
        except Exception as e:
            error_msg = str(e)
            print(f"⚠ Initial load failed: {error_msg[:150]}...")
            
            # Check if it's a Lambda layer issue (CBAM model)
            if "Lambda" in error_msg and "output_shape" in error_msg:
                print(f"  Detected Lambda layer issue (CBAM model)")
                print(f"  Attempting to load with custom Lambda deserialization...")
                
                try:
                    # Custom deserialization for Lambda layers
                    import h5py
                    
                    # Read model architecture from HDF5
                    with h5py.File(model_path, 'r') as f:
                        model_config = f.attrs.get('model_config')
                        if model_config is None:
                            model_config = f.attrs.get('model_architecture')
                        
                        if isinstance(model_config, bytes):
                            model_config = model_config.decode('utf-8')
                        
                        import json
                        model_dict = json.loads(model_config)
                        
                        # Fix Lambda layers by adding output_shape
                        def fix_lambda_layers(config):
                            if isinstance(config, dict):
                                if config.get('class_name') == 'Lambda':
                                    # Add output_shape if missing
                                    if 'output_shape' not in config.get('config', {}):
                                        # For reduce_mean and reduce_max, output keeps spatial dims but changes channels to 1
                                        config['config']['output_shape'] = None
                                        config['config']['output_shape_type'] = 'raw'
                                        config['config']['output_shape_module'] = None
                                
                                for key, value in config.items():
                                    fix_lambda_layers(value)
                            elif isinstance(config, list):
                                for item in config:
                                    fix_lambda_layers(item)
                        
                        fix_lambda_layers(model_dict)
                        
                        # Reconstruct model from fixed config
                        with tf.keras.utils.custom_object_scope(custom_objects):
                            model = tf.keras.models.model_from_json(
                                json.dumps(model_dict),
                                custom_objects=custom_objects
                            )
                        
                        # Load weights
                        model.load_weights(model_path)
                    
                    print(f"✓ Model loaded successfully with Lambda fix")
                    
                except Exception as e3:
                    print(f"✗ Lambda fix failed: {str(e3)[:200]}")
                    print(f"  Skipping this model...")
                    continue
            else:
                # Not a Lambda issue, try with compile=True
                print(f"  Attempting with compile=True...")
                
                try:
                    with tf.keras.utils.custom_object_scope(custom_objects):
                        model = models.load_model(model_path, compile=True)
                        # Recompile to avoid optimizer issues
                        model.compile(optimizer='adam', loss='categorical_crossentropy')
                    print(f"✓ Model loaded successfully with compile=True")
                except Exception as e2:
                    print(f"✗ Error loading model: {str(e2)[:200]}")
                    print(f"   Full error: {e2}")
                    continue
        
        # Evaluate
        metrics, params = evaluate_model_comprehensive(model, X_test, y_test, model_name)
        all_results[model_name] = metrics
        all_params[model_name] = params
        
        # Clean up
        del model
        K.clear_session()
    
    # ========================================================================
    # SAVE RESULTS
    # ========================================================================
    
    print(f"\n{'='*100}")
    print(f"{'SAVING RESULTS':^100s}")
    print(f"{'='*100}\n")
    
    # Create comprehensive DataFrame
    results_data = []
    for model_name, metrics in all_results.items():
        params = all_params.get(model_name, 0)
        for metric in metrics:
            results_data.append({
                'Model': model_name,
                'Parameters': f"{params:,}",
                'Class': metric['class'],
                'Precision': f"{metric['precision']:.4f}",
                'Recall': f"{metric['recall']:.4f}",
                'F1-Score': f"{metric['f1_score']:.4f}",
                'Dice': f"{metric['dice']:.4f}",
                'IoU': f"{metric['iou']:.4f}"
            })
    
    df = pd.DataFrame(results_data)
    
    # Save to CSV
    csv_path = os.path.join(OUTPUT_DIR, "comprehensive_metrics.csv")
    df.to_csv(csv_path, index=False)
    print(f"✓ Saved comprehensive metrics: {csv_path}")
    
    # Create summary table (Overall metrics only)
    summary_data = []
    for model_name, metrics in all_results.items():
        overall = metrics[-1]  # Last entry is overall
        params = all_params.get(model_name, 0)
        summary_data.append({
            'Model': model_name,
            'Parameters': f"{params:,}",
            'Precision': f"{overall['precision']:.4f}",
            'Recall': f"{overall['recall']:.4f}",
            'F1-Score': f"{overall['f1_score']:.4f}",
            'Dice': f"{overall['dice']:.4f}",
            'IoU': f"{overall['iou']:.4f}"
        })
    
    df_summary = pd.DataFrame(summary_data)
    summary_path = os.path.join(OUTPUT_DIR, "overall_metrics_summary.csv")
    df_summary.to_csv(summary_path, index=False)
    print(f"✓ Saved overall summary: {summary_path}")
    
    # Print final summary
    print(f"\n{'='*100}")
    print(f"{'FINAL SUMMARY - OVERALL METRICS':^100s}")
    print(f"{'='*100}\n")
    print(df_summary.to_string(index=False))
    
    print(f"\n{'='*100}")
    print(f"{'✓ EVALUATION COMPLETED!':^100s}")
    print(f"{'='*100}")
    print(f"\nResults saved in: {OUTPUT_DIR}/")
    print(f"  - comprehensive_metrics.csv (per-class + overall)")
    print(f"  - overall_metrics_summary.csv (overall only)")
    print(f"\n{'='*100}\n")

if __name__ == "__main__":
    main()

2025-11-22 07:32:39.354096: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763796759.542849      38 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763796759.601735      38 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered



                          EVALUATING SAVED MODELS - COMPREHENSIVE METRICS                           

Configuring GPU...
✓ GPU found: /physical_device:GPU:0

                                         LOADING TEST DATA                                          

  Loading train data...
    train: 400 samples
  Loading val data...
    val: 400 samples
  Loading test data...
    test: 400 samples
  Total: 1200 samples

Test split created: 400 samples
  Shape: (400, 256, 256, 3)
  Masks shape: (400, 256, 256)
  Mask classes: [0 1 2]


LOADING MODEL: mobilenet_unet_cbam.h5


I0000 00:00:1763796835.388285      38 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


⚠ Initial load failed: Exception encountered when calling Lambda.call().

We could not automatically infer the shape of the Lambda's output. Please specify the `output_s...
  Detected Lambda layer issue (CBAM model)
  Attempting to load with custom Lambda deserialization...
✗ Lambda fix failed: Could not locate class 'Functional'. Make sure custom classes are decorated with `@keras.saving.register_keras_serializable()`. Full object config: {'class_name': 'Functional', 'config': {'name': 'Mob
  Skipping this model...

LOADING MODEL: mobilenet_unet_transformer.h5
✓ Model loaded successfully

EVALUATING: MobileNet-UNet + Transformer
  Total Parameters: 28,800,536
  Generating predictions...


I0000 00:00:1763796845.481526      85 service.cc:148] XLA service 0x7d090000d840 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1763796845.482168      85 service.cc:156]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1763796846.213843      85 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1763796854.302820      85 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.



Precision, Recall, F1 Score per class:
  Background: Precision=0.9994, Recall=0.9992, F1=0.9993
  Disc: Precision=0.8852, Recall=0.9279, F1=0.9061
  Cup: Precision=0.9216, Recall=0.8465, F1=0.8824
Overall Precision: 0.9354
Overall Recall: 0.9245
Overall F1 Score: 0.9293

Dice and IoU per class:
  Background: Dice=0.9993, IoU=0.9986
  Disc: Dice=0.9061, IoU=0.8283
  Cup: Dice=0.8824, IoU=0.7896
Overall Dice: 0.9293
Overall IoU: 0.8722

LOADING MODEL: mobilenet_unet_convex_prior.h5
✓ Model loaded successfully

EVALUATING: MobileNet-UNet + Convex Prior
  Total Parameters: 16,184,003
  Generating predictions...

Precision, Recall, F1 Score per class:
  Background: Precision=0.9993, Recall=0.9993, F1=0.9993
  Disc: Precision=0.8921, Recall=0.9069, F1=0.8994
  Cup: Precision=0.8996, Recall=0.8571, F1=0.8778
Overall Precision: 0.9303
Overall Recall: 0.9211
Overall F1 Score: 0.9255

Dice and IoU per class:
  Background: Dice=0.9993, IoU=0.9986
  Disc: Dice=0.8994, IoU=0.8172
  Cup: Dice=0.877